# 03 · Barrido de estilo: qué distingue a Jardine por partido

**Qué hace.** Compara el estilo **por defecto** del América de Jardine con el de Solari y Ortiz, partido a partido, descontando a la liga: cada métrica del América menos el promedio del resto de la Liga MX en el mismo torneo. Completa al notebook 02 (que mide cómo **reacciona** a los quiebres) y cierra con la **tabla final de qué pasa y qué no** de los dos notebooks.

**Qué mide** (lista fijada antes de ver resultados):
- *Team Match Stats de toda la liga* (ajustadas por la liga): PPDA, distancia defensiva, posesión, directness, % de posesiones en transición, % de centros entre los pases al área, xG de balón parado y contrapresiones.
- *Eventos del América* (sin ajuste por liga, porque solo hay eventos de sus partidos): field tilt y largo de pase.
- *Formaciones* iniciales (descriptivo) y *huella de pases* (los 60 `pass_cluster_id` de StatsBomb): distancia entre huellas con prueba de permutación y criterio por cluster.

**Qué necesita.** Pasos 02–05 corridos (`data/processed/` y `outputs/panel/`) y, para la tabla final, el notebook 02 corrido (`outputs/hallazgos/`).

**Cómo probarlo.** En VS Code, kernel `.venv`, *Run All*. Tarda menos de un minuto.

**Criterio de hallazgo, versión 2** (18 oct). Las tres condiciones del 12 oct se mantienen:
- IC 95 % por bootstrap de partidos sin cruzar 0;
- tamaño ≥ 0.5 DE entre partidos (aquí la escala correcta, porque se comparan partidos completos);
- misma dirección en al menos 4 de los 5 torneos de entrenamiento de Jardine.

Se agrega una cuarta: sobrevivir a **Benjamini-Hochberg** (q ≤ `hallazgo.fdr_q`) dentro de cada familia de pruebas (estilo, huella). La tabla final informa además el q con todas las pruebas juntas.

> IMPOR1: sin el Clausura 2026 (torneo de prueba), ni del América ni de la liga.
>
> IMPOR2: solo agregados. El notebook se sube sin salidas (nbstripout).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import jensenshannon

from ojo.config import DIR_SALIDAS, params
from ojo.datos import almacen
from ojo.framework import juego
from ojo.framework import partidos as fp
from ojo.lentes import hallazgo, minuto
from ojo.salidas.estilo import color

P = params()
EQUIPO = P["equipo"]["team_id"]
HALLAZGO = P["hallazgo"]
RNG = np.random.default_rng(P["simulacion"]["semilla"])
REPETICIONES = P["estadistica"]["bootstrap_repeticiones"]
INCLUIR_PRUEBA = False  # IMPOR1
DIR_FIG, DIR_HALLAZGOS = DIR_SALIDAS / "exploracion", DIR_SALIDAS / "hallazgos"
for d in (DIR_FIG, DIR_HALLAZGOS):
    d.mkdir(parents=True, exist_ok=True)

ENTRENADORES = {d["manager_id"]: d["nombre"].split()[-1] for d in P["predecesores"]}
ENTRENADORES[P["entrenador"]["manager_id"]] = P["entrenador"]["nombre"].split()[-1]
ORDEN = list(ENTRENADORES.values())
NUEVO, PREVIOS = ORDEN[-1], "Predecesores"

# Métricas de Team Match Stats (contexto/api/team_match_stats.md). directness = la de Phases of Play sobre
# todas las posesiones (la columna `directness` solo mira posesiones que terminan en tiro).
TMS = {
    "ppda": ("team_match_ppda", "PPDA (más bajo = presiona más)"),
    "distancia_defensiva": ("team_match_defensive_distance", "Distancia defensiva (m desde su portería)"),
    "posesion": ("team_match_possession", "Posesión"),
    "directness": ("team_match_avg_team_possession_directness", "Directness de las posesiones"),
    "transicion": ("team_match_transition_possession_rate", "% de posesiones en transición"),
    "centros": ("team_match_box_cross_ratio", "% de centros entre los pases al área"),
    "xg_balon_parado": ("team_match_sp_xg", "xG de balón parado"),
    "contrapresiones": ("team_match_counterpressures", "Contrapresiones"),
}
NOMBRES = {k: v[1] for k, v in TMS.items()} | {"field_tilt": "Field tilt", "largo_pase": "Largo de pase (yd)"}

## 1. Partidos de la liga y del América

In [ ]:
liga = fp.con_torneo(fp.validos(almacen.cargar("partidos")))
liga["prueba"] = fp.es_prueba(liga)
if not INCLUIR_PRUEBA:
    liga = liga[~liga["prueba"]]
liga["torneo_id"] = liga["season_id"].astype(str) + "-" + liga["torneo"]
america = fp.con_torneo(fp.del_equipo(liga, EQUIPO))
america = america[america["entrenador_id"].isin(ENTRENADORES)].copy()
america["entrenador"] = america["entrenador_id"].map(ENTRENADORES)
america["grupo"] = np.where(america["entrenador"] == NUEVO, NUEVO, PREVIOS)
TORNEOS_NUEVO = list(america.loc[america["grupo"] == NUEVO].sort_values("fecha")["torneo_corto"].unique())
print(f"Liga: {len(liga):,} partidos en {liga['torneo_id'].nunique()} torneos · América: {len(america)} partidos")
america.groupby("entrenador", sort=False).agg(partidos=("match_id", "size"),
                                              torneos=("torneo_corto", lambda s: ", ".join(s.unique())))

## 2. Estilo ajustado por la liga

Para cada partido del América: su valor menos el promedio de los otros equipos de la liga en el mismo torneo. Así un cambio de toda la liga (por ejemplo, más presión en general) no se confunde con un cambio del entrenador.

In [ ]:
columnas = ["match_id", "team_id"] + [c for c, _ in TMS.values()]
tms = almacen.cargar("estadisticas_equipo", columnas=columnas).merge(liga[["match_id", "torneo_id"]], on="match_id")
tms = tms.rename(columns={c: k for k, (c, _) in TMS.items()})
resto = tms[tms["team_id"] != EQUIPO].groupby("torneo_id")[list(TMS)].mean()
del_america = tms[tms["team_id"] == EQUIPO].set_index("match_id")
ajustado = del_america[list(TMS)] - resto.reindex(del_america["torneo_id"]).to_numpy()
estilo = america[["match_id", "entrenador", "grupo", "torneo_corto"]].join(ajustado, on="match_id")

# Del panel por minuto (05) y de los eventos: field tilt y largo de pase del América, sin ajuste por liga.
panel = minuto.cargar_panel("minuto").query("match_id in @estilo.match_id")
tercio = panel.groupby(["match_id", "lado"])["pases_ultimo_tercio"].sum().unstack()
estilo["field_tilt"] = estilo["match_id"].map(tercio["propio"] / (tercio["propio"] + tercio["rival"]))
eventos = almacen.cargar("eventos", estilo["match_id"], columnas=["match_id", "equipo_id", "tipo", "patron", "x", "y",
                                                                  "fin_x", "fin_y", "pase_cluster_id", "pase_cluster"])
pases = eventos[(eventos["equipo_id"] == EQUIPO) & (eventos["tipo"] == "Pass") & juego.es_juego_abierto(eventos)]
largo = np.hypot(pases["fin_x"] - pases["x"], pases["fin_y"] - pases["y"])
estilo["largo_pase"] = estilo["match_id"].map(largo.groupby(pases["match_id"]).mean())
METRICAS = list(TMS) + ["field_tilt", "largo_pase"]
estilo.groupby("entrenador", sort=False)[METRICAS].mean().T.reindex(columns=ORDEN).round(3)

## 3. Jardine frente a sus predecesores

Diferencia de medias (Jardine − predecesores), bootstrap de partidos independiente en cada grupo. La condición 3 compara cada torneo de Jardine con el promedio de los predecesores. La DE es la de la métrica entre todos los partidos del América usados.

In [ ]:
def comparar(tabla: pd.DataFrame, metricas: list[str]) -> pd.DataFrame:
    # Criterio de hallazgo para la diferencia Jardine − predecesores de cada métrica por partido.
    filas = []
    for m in metricas:
        datos = tabla[["match_id", "grupo", "torneo_corto", m]].dropna()
        nuevo, previo = datos[datos["grupo"] == NUEVO], datos[datos["grupo"] == PREVIOS]
        medias = hallazgo.medias_bootstrap(nuevo, m, RNG) - hallazgo.medias_bootstrap(previo, m, RNG)
        bajo, alto = hallazgo.intervalo(medias)
        diferencia = nuevo[m].mean() - previo[m].mean()
        por_torneo = nuevo.groupby("torneo_corto")[m].mean() - previo[m].mean()
        misma = hallazgo.direcciones(por_torneo, diferencia)
        filas.append({"metrica": m, "nuevo": nuevo[m].mean(), "previos": previo[m].mean(), "diferencia": diferencia,
                      "ic_bajo": bajo, "ic_alto": alto, "torneos_misma_direccion": misma,
                      "muestra_chica": min(nuevo["match_id"].nunique(), previo["match_id"].nunique()) < HALLAZGO["partidos_aviso"],
                      "p": hallazgo.p_bootstrap(medias),
                      **hallazgo.evaluar(diferencia, bajo, alto, datos[m].std(), misma)})
    return hallazgo.con_fdr(pd.DataFrame(filas))


barrido = comparar(estilo, METRICAS)
barrido.to_parquet(DIR_HALLAZGOS / "estilo.parquet", index=False)
print(f"Pruebas: {len(barrido)} · condiciones 1–3: {barrido['pasa_1a3'].sum()} · v2 (con FDR): {barrido['pasa'].sum()}")
barrido.assign(metrica=barrido["metrica"].map(NOMBRES)).round(3)

**Control del largo de pase.** El field tilt y el largo de pase no se pueden ajustar por la liga (solo hay eventos de los partidos del América). Como control, se mide el largo de pase **de los rivales en esos mismos partidos**: si también bajó, la diferencia puede ser una tendencia de la liga y no de Jardine. No entra a la tabla final.

In [ ]:
pases_rival = eventos[(eventos["equipo_id"] != EQUIPO) & (eventos["tipo"] == "Pass") & juego.es_juego_abierto(eventos)]
largo_rival = np.hypot(pases_rival["fin_x"] - pases_rival["x"], pases_rival["fin_y"] - pases_rival["y"])
estilo["largo_pase_rival"] = estilo["match_id"].map(largo_rival.groupby(pases_rival["match_id"]).mean())
comparar(estilo, ["largo_pase_rival"])[["metrica", "nuevo", "previos", "diferencia", "ic_bajo", "ic_alto", "tamano_de"]].round(3)

In [ ]:
fig, eje = plt.subplots(figsize=(8, 5))
orden = barrido.sort_values("tamano_de")
y = np.arange(len(orden))
for i, f in enumerate(orden.itertuples()):
    de = f.diferencia / f.tamano_de if f.tamano_de else np.nan
    eje.plot([f.ic_bajo / de, f.ic_alto / de], [i, i], color=color("gris-texto"), lw=2)
    eje.plot(f.tamano_de, i, "o", ms=9, color=color("dorado") if f.pasa else color("azul"))
eje.axvline(0, color="0.5", lw=1)
for limite in (-HALLAZGO["tamano_min_de"], HALLAZGO["tamano_min_de"]):
    eje.axvline(limite, color="0.7", lw=1, ls=":")
eje.set_yticks(y, orden["metrica"].map(NOMBRES))
eje.set_xlabel(f"{NUEVO} − predecesores (DE entre partidos; dorado = pasa el criterio)")
fig.savefig(DIR_FIG / "03_estilo.png", dpi=130, bbox_inches="tight")
plt.show()

## 4. Formaciones iniciales (descriptivo)

Proporción de partidos que cada entrenador empezó con cada formación (`Starting XI` de StatsBomb). No entra al criterio: es una variable de categorías, se muestra como contexto.

In [ ]:
inicio = almacen.cargar("eventos", estilo["match_id"], columnas=["match_id", "equipo_id", "tipo", "formacion"])
inicio = inicio[(inicio["tipo"] == "Starting XI") & (inicio["equipo_id"] == EQUIPO)].merge(america[["match_id", "entrenador"]])
inicio["formacion"] = inicio["formacion"].astype(int).astype(str)
formaciones = pd.crosstab(inicio["formacion"], inicio["entrenador"], normalize="columns").reindex(columns=ORDEN)
formaciones[formaciones.max(axis=1) >= 0.05].sort_values(NUEVO, ascending=False).round(2)

## 5. Huella de pases

Para cada partido, la fracción de los pases del América en juego abierto que cae en cada uno de los 60 clusters de StatsBomb. Se compara la huella media de Jardine con la de sus predecesores con la distancia de Jensen-Shannon (0 = idénticas) y una **prueba de permutación**: si se reparten al azar los partidos entre los dos grupos, ¿qué tan seguido sale una distancia así de grande? Para leerla, se compara con la distancia entre los propios torneos de Jardine.

In [ ]:
con_cluster = pases.dropna(subset=["pase_cluster_id"])
huella = pd.crosstab(con_cluster["match_id"], con_cluster["pase_cluster_id"].astype(int), normalize="index")
huella = huella.reindex(columns=range(60), fill_value=0.0)
grupo = america.set_index("match_id").loc[huella.index, "grupo"].to_numpy()
torneo = america.set_index("match_id").loc[huella.index, "torneo_corto"].to_numpy()
X = huella.to_numpy()


def distancia(etiquetas: np.ndarray) -> float:
    return float(jensenshannon(X[etiquetas == NUEVO].mean(axis=0), X[etiquetas != NUEVO].mean(axis=0)))


observada = distancia(grupo)
permutadas = np.array([distancia(RNG.permutation(grupo)) for _ in range(REPETICIONES)])
p_valor = (1 + (permutadas >= observada).sum()) / (1 + REPETICIONES)
print(f"Distancia {NUEVO}–predecesores: {observada:.4f} (permutación: p = {p_valor:.4f}; "
      f"percentil 95 al azar: {np.quantile(permutadas, .95):.4f})")


**Estabilidad de la huella.** La distancia entre dos grupos chicos siempre es mayor por puro ruido, así que cada par de torneos de Jardine se compara con su propia prueba de permutación: ¿sus huellas difieren más de lo que da el azar con esos mismos partidos?

In [ ]:
def p_permutacion(a: np.ndarray, b: np.ndarray) -> tuple[float, float]:
    # Distancia de Jensen-Shannon entre dos grupos de partidos y su p por permutación.
    juntos, n = np.vstack([a, b]), len(a)
    obs = float(jensenshannon(a.mean(axis=0), b.mean(axis=0)))
    azar = [jensenshannon(juntos[i[:n]].mean(axis=0), juntos[i[n:]].mean(axis=0))
            for i in (RNG.permutation(len(juntos)) for _ in range(REPETICIONES))]
    return obs, (1 + sum(d >= obs for d in azar)) / (1 + REPETICIONES)


pares = []
for i, a in enumerate(TORNEOS_NUEVO):
    for b in TORNEOS_NUEVO[i + 1:]:
        obs, p = p_permutacion(X[(grupo == NUEVO) & (torneo == a)], X[(grupo == NUEVO) & (torneo == b)])
        pares.append({"torneos": f"{a} – {b}", "distancia": obs, "p": p})
pares = pd.DataFrame(pares)
print(f"Pares de torneos de {NUEVO} que difieren más que el azar (p < 0.05): {(pares['p'] < 0.05).sum()} de {len(pares)}")
pares.round(4)

In [ ]:
etiquetas = (con_cluster.dropna(subset=["pase_cluster"]).groupby(con_cluster["pase_cluster_id"].astype(int))["pase_cluster"]
             .agg(lambda s: s.mode().iloc[0]))
por_cluster = huella.add_prefix("c").reset_index().merge(america[["match_id", "grupo", "torneo_corto"]], on="match_id")
clusters = comparar(por_cluster, [f"c{i}" for i in range(60)])
clusters["cluster"] = clusters["metrica"].str[1:].astype(int)
clusters["etiqueta"] = clusters["cluster"].map(etiquetas)
clusters.to_parquet(DIR_HALLAZGOS / "huella.parquet", index=False)
print(f"Clusters probados: {len(clusters)} · condiciones 1–3: {clusters['pasa_1a3'].sum()} · v2 (con FDR): {clusters['pasa'].sum()} "
      f"(por azar se esperan ~{0.05 * len(clusters):.0f} en la condición 1)")
clusters[clusters["pasa"]].sort_values("tamano_de")[["cluster", "etiqueta", "previos", "nuevo", "diferencia",
                                                     "tamano_de", "torneos_misma_direccion"]].round(4)

## 6. Tabla final: qué pasa y qué no

Junta las pruebas de los notebooks 02 y 03 con el mismo criterio. Cada fila de abajo es un candidato a principio; los que tienen `muestra_chica` se leen con más cautela.

In [ ]:
archivos = {
    "Quiebre con Jardine": ("quiebres_jardine", "efecto"),
    "Quiebre: Jardine − predecesores": ("quiebres_contra_predecesores", "diferencia"),
    "Agresividad (Jardine)": ("agresividad", "coef_por_DE_presion"),
    "Estilo: Jardine − predecesores": ("estilo", "diferencia"),
    "Huella: Jardine − predecesores": ("huella", "diferencia"),
}
partes = []
for analisis, (nombre, valor) in archivos.items():
    ruta = DIR_HALLAZGOS / f"{nombre}.parquet"
    if not ruta.exists():
        print(f"Falta {ruta.name}: corre el notebook 02.")
        continue
    t = pd.read_parquet(ruta)
    if nombre == "agresividad":
        t = t[t["grupo"] == NUEVO].assign(metrica=t["respuesta"] + " · " + t["momento"], muestra_chica=False)
    if nombre == "huella":
        t = t.assign(metrica="cluster " + t["cluster"].astype(str) + " · " + t["etiqueta"].fillna(""))
    prueba = (t["tipo"] + " · " + t["lado"] + " · " + t["metrica"]) if "tipo" in t else t["metrica"].map(lambda m: NOMBRES.get(m, m))
    partes.append(pd.DataFrame({"análisis": analisis, "prueba": prueba, "valor": t[valor], "ic_bajo": t["ic_bajo"],
                                "ic_alto": t["ic_alto"], "tamano_de": t["tamano_de"],
                                "torneos": t["torneos_misma_direccion"], "muestra_chica": t["muestra_chica"].astype(bool),
                                "p": t["p"], "q": t["q"], "pasa_1a3": t["pasa_1a3"].astype(bool), "pasa": t["pasa"].astype(bool)}))
final = pd.concat(partes, ignore_index=True)
final["q_global"] = hallazgo.fdr_bh(final["p"])  # todas las pruebas de 02 y 03 juntas: la lectura más estricta
final["pasa_global"] = final["pasa"] & (final["q_global"] <= HALLAZGO["fdr_q"])
final.to_parquet(DIR_HALLAZGOS / "tabla_final.parquet", index=False)
conteo = final.groupby("análisis", sort=False).agg(pruebas=("pasa", "size"), pasan_1a3=("pasa_1a3", "sum"), pasan=("pasa", "sum"),
                                                   pasan_q_global=("pasa_global", "sum"),
                                                   pasan_con_10_partidos=("pasa", lambda s: int((s & ~final.loc[s.index, "muestra_chica"]).sum())))
conteo["por azar en la condición 1 (≈5 %)"] = (conteo["pruebas"] * 0.05).round(1)
conteo

In [ ]:
final[final["pasa"]].sort_values(["análisis", "muestra_chica", "tamano_de"], key=lambda s: s.abs() if s.name == "tamano_de" else s,
                                  ascending=[True, True, False]).round(3)

## Lectura

Corrida del 12 oct (AP21–AP25, sin el CL26); el punto 6 se actualizó el 18 oct con el criterio v2. Los números cambian si cambian los datos o `params.yaml`.

1. **Estilo por defecto: solo el largo de pase pasa.** Con Jardine, los pases en juego abierto miden 1.5 yd menos que con sus predecesores (20.5 frente a 22.0; 0.73 DE; 5 de 5 torneos). No parece tendencia de la liga: en esos mismos partidos, el pase de los rivales casi no cambia (−0.4 yd, IC que cruza 0).
2. **Cerca, pero no pasan.** PPDA: frente a la liga, el América de Jardine presiona **menos** que el de sus predecesores (+1.9 de PPDA; IC sin 0, pero 0.35 DE y 3 de 5 torneos). Centros: −3.9 puntos en el % de pases al área que son centros (5 de 5, pero 0.37 DE). Posesiones en transición: −1 punto (0.29 DE). La presión alta no aparece como identidad de Jardine, igual que en la hipótesis de agresividad del notebook 02.
3. **Huella de pases distinta.** La huella media de Jardine se separa de la de sus predecesores más de lo que da el azar (Jensen-Shannon 0.059; permutación p = 0.0005). Los 6 clusters que pasan cuentan lo mismo que el largo de pase: menos pases largos hacia adelante desde el centro del tercio defensivo y desde la derecha del medio, y más pases cortos a ras de piso en el tercio defensivo y pases cortos hacia atrás en el tercio final por izquierda. Por azar se esperaban unos 3. Candidato a principio: *salida corta y paciente, que cambia el pelotazo por circulación corta*.
4. **La huella evoluciona.** 5 de 10 pares de torneos de Jardine difieren más que el azar, sobre todo el AP25 frente a los primeros. La dirección de los principios por partido se sostiene (5 de 5 en el largo de pase) aunque la huella completa cambie.
5. **Formaciones (descriptivo).** El 4-2-3-1 es la base de los tres. Jardine usa más línea de tres (3-4-3 en 11 % de sus partidos y 3-4-2-1 en 6 %) que Ortiz (0 %).
6. **Tabla final (02 + 03, criterio v2).** De 636 pruebas pasan 30; las mismas sobreviven al FDR con todas las pruebas juntas. En la v1 pasaban 69. Principios para la entrega:
    - *Sale jugando corto*: largo de pase −1.5 yd, 5 de 5, q = 0.01, más 6 clusters de la huella.
    - *Con uno más, hunde al rival*: tras una expulsión a favor, el bloque del rival 19.5 yd más atrás que contra los predecesores y menos transiciones en contra.
    - *Una lesión no lo frena*: más presión tras una sustitución por lesión.
    - *Tras recibir gol, el rival no lo aprieta*: el rival contrapresiona menos.
    - Alerta: tras sus cambios de sistema baja el xG.
    - Lo que **no** es: un equipo de presión alta (PPDA y agresividad no pasan). Tampoco pasan con v2 *ganando, cierra la transición* ni *sus cambios ganan espacio arriba*.